In [18]:
# Install core dependencies for Speech-to-Speech pipeline
# !pip install -q transformers torch torchaudio librosa soundfile noisereduce
# Ensure setuptools is present to avoid ModuleNotFoundError
# !pip install -q setuptools

In [32]:
import os
import torch
import librosa
import numpy as np
import soundfile as sf
import noisereduce as nr
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    LogitsProcessorList,
    SequenceBiasLogitsProcessor,
    VitsModel,
    AutoTokenizer
)

# =========================
# 1. CONFIGURATION
# =========================
# Upload your 'extracted_sample.wav' and 'student_voice_ref.wav' to the Colab file browser
INPUT_AUDIO = "extracted_sample.wav"
VOICE_REF = "student_voice_ref.wav"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Using T4 GPU if available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# 2. TASK 1.3: DENOISING
# =========================
def denoise_audio(path):
    print("🔹 Task 1.3: Denoising Classroom Audio...")
    y, sr = librosa.load(path, sr=16000)
    # Stationary noise reduction for clean student-built implementation
    reduced = nr.reduce_noise(y=y, sr=sr, prop_decrease=0.8)
    output_path = os.path.join(OUTPUT_DIR, "denoised_input.wav")
    sf.write(output_path, reduced, sr)
    return output_path

# =========================
# 3. TASK 1.2: CONSTRAINED TRANSCRIPTION
# =========================
import gc

def transcribe_with_bias(audio_path):
    print("🔹 Task 1.2: Transcribing with Manual Chunking & Sequence Bias...")

    # 1. Load Model and Processor (Using medium to prevent OOM)
    model_id = "openai/whisper-medium"
    processor = WhisperProcessor.from_pretrained(model_id)

    model = WhisperForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16
    ).to(DEVICE)

    # 2. Setup Sequence Bias using LogitsProcessor (The Hugging Face Way)
    tech_terms = ["stochastic", "cepstrum", "archaeology", "excavation", "buddhist"]

    sequence_bias_dict = {}
    for term in tech_terms:
        token_ids = processor.tokenizer.encode(f" {term}", add_special_tokens=False)
        # SequenceBiasLogitsProcessor requires tuples of token IDs as keys
        sequence_bias_dict[tuple(token_ids)] = 15.0

    # Initialize the Logits Processor List
    bias_processor = SequenceBiasLogitsProcessor(sequence_bias=sequence_bias_dict)
    logits_processor = LogitsProcessorList([bias_processor])

    # 3. Load Audio
    speech, sr = librosa.load(audio_path, sr=16000)

    # 4. Manual Chunking (30-second windows)
    chunk_size = 30 * sr  # 30 seconds
    full_transcript = []

    print(f"   Processing {len(speech) // chunk_size + 1} chunks...")

    for i in range(0, len(speech), chunk_size):
        chunk = speech[i : i + chunk_size]

        # Cast the extracted features to match the model's float16 data type
        input_features = processor(
            chunk,
            sampling_rate=sr,
            return_tensors="pt"
        ).input_features.to(DEVICE, dtype=model.dtype)

        # 5. Generate with Explicit Logits Processor
        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                logits_processor=logits_processor, # <--- BUG FIX IS HERE
                return_timestamps=True
            )

        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        full_transcript.append(transcription)

        # Aggressive Garbage Collection after each chunk to prevent OOM
        del input_features
        del predicted_ids
        torch.cuda.empty_cache()
        gc.collect()

    return " ".join(full_transcript)


# =========================
# 4. TASK 3.3: MARATHI SYNTHESIS (LRL)
# =========================
def synthesize_marathi(text):
    print("🔹 Task 3.3: Zero-Shot Marathi Synthesis (MMS)...")
    # VITS architecture supports generative speech
    model_id = "facebook/mms-tts-mar"
    model = VitsModel.from_pretrained(model_id).to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output = model(**inputs).waveform

    output_path = os.path.join(OUTPUT_DIR, "output_LRL_cloned.wav")
    sf.write(output_path, output.cpu().numpy().squeeze(), model.config.sampling_rate)
    return output_path


In [ ]:

# =========================
# 5. MAIN EXECUTION
# =========================
def main():
    if not os.path.exists(INPUT_AUDIO):
        print(f"Upload {INPUT_AUDIO} to Colab first!")
        return

    # Process Pipeline
    clean_audio = denoise_audio(INPUT_AUDIO)

    # Transcription results
    transcript_text = transcribe_with_bias(clean_audio)

    # BUG FIX: transcript_text is already a string, no need for ['text']
    print(f"\nFull Transcript:\n{transcript_text}")

    # Part II/III Placeholder: Use your manual Marathi translation here
    marathi_text = "नमस्कार, आपण एम-टेक डेटा इंजिनिअरिंग प्रकल्पावर काम करत आहोत."
    synthesize_marathi(marathi_text)

    print(f"\nPipeline Complete. Download files from the '{OUTPUT_DIR}' folder.")

if __name__ == "__main__":
    main()

🔹 Task 1.3: Denoising Classroom Audio...
🔹 Task 1.2: Transcribing with Manual Chunking & Sequence Bias...


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

   Processing 23 chunks...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA


📝 Full Transcript:
 और तीश्रा मैं कहता था, मेरे लिये मैं कुछ नहीं करूँगा, आज मुझे 24 साल वोर गये हैं, इतने लमबे काल्खन से मैं, हेड़ अप गौर्मेंट के रूप में देशभासी हैं और में जे काम दिया हैं, मेरे इन तीन कसोटियों पर मैं अपने आपको तोल करके रखा हुआ हैं, और मैं उसको करता हूँ। तो मुझे मेरा एक इस्परेशन बड़ा बिर्यालोको की सेवा उनके अपने पराइज़ाइन पर पर्भाज़ाइन पर पर्भाज़ाइन पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर पर  अस्पिरेशन, उनकी आवशकता, जितनी महनत कर सकूँ, जितना कर सकूँ, मैं करने के मूद में हैं। वाइक भी मेरी उर्जा वो ही हैं। एक इंजीनियर और गणित प्रेमी होने के नाते, मुझे पूछना होगा, श्रीनिवास रामानुजन, एक सदी पहले के भार्तिय गणितअग्य थे, उन्हें इतिहास के सबसे महानतम गणितअग्यों में से एक माना जाता है, अपने आप सब सीखा, गरीबी में 

...

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/918 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]


✅ Pipeline Complete. Download files from the 'outputs' folder.
